# 19. Generators & Lazy Evaluation
Exhaustive guide to iterator protocols, generator yield, generator frames memory, coroutines, sub-generators delegating, and pipelines.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for combined analysis questions at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. The Iterator Protocol
**Explanation**: Requires implementing __iter__ and __next__.

**Syntax**:
```python
def __next__(self): return next_value
```



In [ ]:
transaction_iterator = iter([1])
print(next(transaction_iterator))

### 2. Generator Functions & yield
**Explanation**: Yields control suspension back to parent.

**Syntax**:
```python
yield value
```



In [ ]:
def suspended_frame_generator(): yield 1
print(next(suspended_frame_generator()))

### 3. Stateful Generator frame memory
**Explanation**: Generators suspend execution frame states dynamically, conserving memory.

**Syntax**:
```python
# state frames behaviors
```

**Visual Explanation (Data with Baraa Style)**:
```mermaid
graph TD
    gen[Generator yield Call] -->|suspends frame state| suspended(GEN_SUSPENDED)
    next[next Call] -->|resumes execution frame state| resumed(GEN_RUNNING)
```


In [ ]:
def stateful_generator():
    running_state = 10
    yield running_state
    running_state += 10
    yield running_state
generator_instance = stateful_generator()
next(generator_instance)
print('Suspended state:', next(generator_instance))

### 4. Generator Expressions
**Explanation**: Inline lazy expressions.

**Syntax**:
```python
generator_object = (expression for item in iterable)
```



In [ ]:
generator_expression_object = (x for x in range(3))
print(next(generator_expression_object))

### 5. Memory comparisons lists vs generator
**Explanation**: Compare memory consumption between lists and generator expressions.

**Syntax**:
```python
import sys
sys.getsizeof(generator_object)
```



In [ ]:
import sys
payout_list = [x for x in range(1000)]
generator_expression_object = (x for x in range(1000))
print('List:', sys.getsizeof(payout_list), 'bytes | Gen:', sys.getsizeof(generator_expression_object), 'bytes')

### 6. Coroutine send() inputs
**Explanation**: Send values into generator frames using .send().

**Syntax**:
```python
received_value = yield yield_value
generator_object.send(value)
```



In [ ]:
def coroutine_function():
    input_value = yield 'Started'
    yield f'Received: {input_value}'
coroutine_instance = coroutine_function()
print(next(coroutine_instance))
print(coroutine_instance.send(100))

### 7. Raising exceptions inside generators stack
**Explanation**: Raises exceptions inside generators.

**Syntax**:
```python
generator_object.throw(ExceptionClass)
```



In [ ]:
def error_yielding_generator():
    try: yield 1
    except ValueError: yield 'Caught'
generator_instance = error_yielding_generator()
next(generator_instance)
print(generator_instance.throw(ValueError))

### 8. Closing active generator frames state
**Explanation**: Closes generator frames early.

**Syntax**:
```python
generator_object.close()
```



In [ ]:
def cleanup_generator():
    try: yield 1
    finally: print('Closed')
generator_instance = cleanup_generator()
next(generator_instance)
generator_instance.close()

### 9. Yielding from sub-generators
**Explanation**: Delegate generation to nested generators using `yield from`.

**Syntax**:
```python
yield from sub_generator
```



In [ ]:
def sub_generator(): yield 1
def parent_generator(): yield from sub_generator()
print(list(parent_generator()))

### 10. Iterating infinite lazy generator streams
**Explanation**: Generate infinite sequences on-demand.

**Syntax**:
```python
while True: yield counter
```



In [ ]:
def infinite_counter_generator():
    counter = 0
    while True: yield counter; counter += 1
generator_instance = infinite_counter_generator()
print(next(generator_instance), next(generator_instance))

### 11. Infinite generators with exit conditions
**Explanation**: Exits infinite generators dynamically.

**Syntax**:
```python
if exit_condition: return
```



In [ ]:
def self_limiting_generator():
    yield 1
    return
print(list(self_limiting_generator()))

### 12. Reusable generators custom classes wrapper
**Explanation**: Wrap generators inside classes to make them reusable.

**Syntax**:
```python
class ReusableGenerator:
    def __iter__(self): yield values
```



In [ ]:
class ReusableGenerator:
    def __iter__(self):
        yield 1
        yield 2
reusable_instance = ReusableGenerator()
print(list(reusable_instance), list(reusable_instance))

### 13. Checking generator inspection states
**Explanation**: Queries active generator states using the inspect module.

**Syntax**:
```python
import inspect
inspect.getgeneratorstate(generator_object)
```



In [ ]:
import inspect
def dummy_generator(): yield 1
generator_instance = dummy_generator()
print('State:', inspect.getgeneratorstate(generator_instance))
next(generator_instance)
print('State after next:', inspect.getgeneratorstate(generator_instance))


### 14. Yielding nested list structures recursively
**Explanation**: Recursively yields nested collections.

**Syntax**:
```python
def flatten_nested_lists(nested_list):
    for item in nested_list:
        yield from flatten_nested_lists(item)
```



In [ ]:
def flatten_nested_lists(nested_list):
    for item in nested_list:
        if isinstance(item, list): yield from flatten_nested_lists(item)
        else: yield item
print(list(flatten_nested_lists([1, [2, 3]])))

### 15. Generator pipelines
**Explanation**: Chain generators to process data pipelines lazily.

**Syntax**:
```python
generator_stage_two = (stage_two_func(x) for x in generator_stage_one)
```



In [ ]:
generator_stage_one = (x for x in range(5))
generator_stage_two = (x * 2 for x in generator_stage_one if x % 2 == 0)
print('Chained pipeline output:', list(generator_stage_two))


## Section 3: Fintech Interview Questions

### Q1: Write a generator pipeline that yields transaction amounts for the first 100 rows, filters out entries < $100, and yields the tax-adjusted amount (amount * 0.18).

In [ ]:
# Solution:
def read_amts(path):
    with open(path, 'r') as f:
        f.readline()
        for _ in range(100):
            row = f.readline().strip().split(',')
            yield float(row[3]) if row[3] not in ('', 'NaN') else 0.0

amts = read_amts(csv_path)
filtered_amts = (a for a in amts if a >= 100.0)
taxed_amts = (round(a * 0.18, 2) for a in filtered_amts)
print('Sample taxed amounts:', [next(taxed_amts) for _ in range(5)])


### Q2: Show generator inspection states dynamically while stepping through generator execution phases.

In [ ]:
# Solution:
import inspect
def audit_gen():
    yield 'CHECK_1'
    yield 'CHECK_2'

ag = audit_gen()
print('State before execution:', inspect.getgeneratorstate(ag))
next(ag)
print('State during execution:', inspect.getgeneratorstate(ag))
